# FOXF1_bead — 07_fixed_quantification

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 07 Fixed Quantification

## Future Development Note

Current DAPI masking and downstream quantification operate on the **fixed small-image files**.
A future improvement worth testing is to perform masking and/or quantification in **large-image space**, since the large fields can contain real cell-containing regions that are truncated in the small images.
This is especially relevant for peripheral / monolayer cells that may be cut off by the small FOV.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

MPLCONFIGDIR = ROOT / ".matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("ROOT:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from scripts import pipeline_common as common
from scripts import live_fixed_quantification as lfq
from scipy.ndimage import gaussian_filter

## Paths And Configuration

In [ ]:
POSITION_MANIFEST = ROOT / "results/manifests/analysis_position_manifest.tsv"
FIXED_SMALL_CENTROIDS = ROOT / "results/annotations/well_centroids_fixed_small_mapped.tsv"

OUT_DIR = ROOT / "results/measurements/live_fixed_small"
QC_DIR = ROOT / "results/qc/07_fixed_quantification"
for path in [OUT_DIR, QC_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FIXED_COHORT_ID = "2026-01-22_day2_fix"
REPRESENTATIVE_POSITIONS = ["1-1", "1-2", "1-3", "2-6", "3-2", "3-3", "4-5", "5-5", "6-6"]
DIST_BIN_UM = 35.0
HIST_SMOOTH_SIGMA_BINS = 6.0
DAPI_GATE_N_SIGMA = 4.0
DAPI_GATE_SWEEP_POSITIONS = ["2-6", "3-3", "4-5", "5-5", "6-6"]
DAPI_GATE_SWEEP_N_SIGMA = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0]
FULL_HIST_UPPER_QUANTILE = 0.99
NULL_PEAK_LEFT_SIGMA = 4.0
NULL_PEAK_RIGHT_SIGMA = 4.0
TRACE_PLOT_YMIN = -0.5
TRACE_YLIM_SUPPORT_FRACTION = 2.0 / 3.0
TRACE_YLIM_UPPER_QUANTILE = 0.95
TRACE_YLIM_UPPER_PAD = 0.10

MANUAL_CHANNEL_EXCLUSION_SPECS = {
    "tagyfp": {
        "4-5": [
            {"shape": "ellipse", "cx": 40.0, "cy": 40.0, "rx": 540.0, "ry": 410.0, "reason": "upper_left_out_of_focus_field_artifact"},
        ],
        "6-4": [
            {"shape": "ellipse", "cx": 770.0, "cy": 950.0, "rx": 495.0, "ry": 495.0, "reason": "lower_right_out_of_focus_circle_artifact"},
        ],
    },
    "sox2": {},
    "t": {},
}

MANUAL_MEASUREMENT_POSITION_EXCLUSIONS = {
    "fixed_tagyfp_bgz_over_dapi_gate": ["4-5"],
}

CANDIDATE_ARTIFACT_REVIEW = {
    "fixed_tagyfp_bgz_over_dapi_gate": ["1-2", "3-2", "4-5", "4-6", "2-4", "5-5", "6-4"],
    "fixed_sox2_bgz_over_dapi_gate": ["3-3", "2-6", "5-2"],
    "fixed_t_bgz_over_dapi_gate": ["3-3", "4-2", "4-5"],
}

MEASUREMENT_ORDER = [
    "fixed_tagyfp_bgz_over_dapi_gate",
    "fixed_sox2_bgz_over_dapi_gate",
    "fixed_t_bgz_over_dapi_gate",
]
MEASUREMENT_LABELS = {
    "fixed_tagyfp_bgz_over_dapi_gate": "FOXF1-YFP (fixed) / DAPI",
    "fixed_sox2_bgz_over_dapi_gate": "SOX2 / DAPI",
    "fixed_t_bgz_over_dapi_gate": "T / DAPI",
}
MEASUREMENT_COLORS = {
    "fixed_tagyfp_bgz_over_dapi_gate": "#2ca25f",
    "fixed_sox2_bgz_over_dapi_gate": "#f16913",
    "fixed_t_bgz_over_dapi_gate": "#8c2d8f",
}
CHANNEL_SPECS = [
    {"key": "dapi", "label": "DAPI", "keywords": ["dapi"], "for_ratio": False},
    {"key": "tagyfp", "label": "FOXF1-YFP", "keywords": ["tagyfp", "foxf1", "yfp"], "for_ratio": True},
    {"key": "sox2", "label": "SOX2", "keywords": ["568", "alexa fluor 568"], "for_ratio": True},
    {"key": "t", "label": "T", "keywords": ["647", "alexa fluor 647"], "for_ratio": True},
]

FIXED_GLOBAL_BG_TSV = OUT_DIR / "fixed_global_background_params.tsv"
FIXED_DAPI_GATE_SUMMARY_TSV = OUT_DIR / "fixed_dapi_gate_summary.tsv"
FIXED_RATIO_PIXEL_BIN_STATS_TSV = OUT_DIR / "fixed_ratio_pixel_bin_stats.tsv"
FIXED_RATIO_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_ratio_distance_trace_across_images.tsv"
FIXED_RATIO_TRACE_ALL_PIXELS_TSV = OUT_DIR / "fixed_ratio_distance_trace_all_pixels_merged.tsv"
FIXED_RATIO_TRACE_EQUAL_SUPPORT_TSV = OUT_DIR / "fixed_ratio_distance_trace_equal_support.tsv"
FIXED_RATIO_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_ratio_distance_trace_equal_support_across_images.tsv"
FIXED_MEASUREMENT_SUMMARY_TXT = OUT_DIR / "fixed_measurement_summary.txt"
FIXED_RATIO_ANALYSIS_PIXELS_NPZ = OUT_DIR / "fixed_ratio_analysis_pixels.npz"

FIXED_GLOBAL_BG_FITS_PNG = QC_DIR / "fixed_global_background_fits.png"
FIXED_DAPI_GATE_SWEEP_PNG = QC_DIR / "fixed_dapi_gate_sigma_sweep_examples.png"
FIXED_DAPI_GATE_SWEEP_VALUES_PNG = QC_DIR / "fixed_dapi_gate_sigma_sweep_removed_values.png"
FIXED_DAPI_GATE_DEBUG_PNG = QC_DIR / "fixed_dapi_gate_debug_examples.png"
FIXED_CANDIDATE_ARTIFACT_REVIEW_PNG = QC_DIR / "fixed_candidate_artifact_review.png"
FIXED_MANUAL_EXCLUSION_APPROVAL_PNG = QC_DIR / "fixed_manual_exclusion_approval_qc.png"
FIXED_MANUAL_EXCLUSION_SUMMARY_TSV = OUT_DIR / "fixed_manual_exclusion_summary.tsv"
FIXED_MANUAL_EXCLUSION_QC_PNG = QC_DIR / "fixed_manual_exclusion_qc.png"
FIXED_RATIO_REPRESENTATIVE_PNG = QC_DIR / "fixed_ratio_representative_images.png"
FIXED_RATIO_TRACE_COMPARISON_PNG = QC_DIR / "fixed_ratio_distance_trace_comparison.png"
FIXED_RATIO_TRACE_FIXED_WIDTH_PNG = QC_DIR / "fixed_ratio_distance_plot_fixed_width.png"
FIXED_RATIO_TRACE_EQUAL_SUPPORT_PNG = QC_DIR / "fixed_ratio_distance_plot_equal_support.png"
FIXED_FOXF1_PIXEL_BIN_STATS_TSV = OUT_DIR / "fixed_foxf1_pixel_bin_stats.tsv"
FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_foxf1_distance_trace_across_images.tsv"
FIXED_FOXF1_TRACE_ALL_PIXELS_TSV = OUT_DIR / "fixed_foxf1_distance_trace_all_pixels_merged.tsv"
FIXED_FOXF1_TRACE_EQUAL_SUPPORT_TSV = OUT_DIR / "fixed_foxf1_distance_trace_equal_support.tsv"
FIXED_FOXF1_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_foxf1_distance_trace_equal_support_across_images.tsv"
FIXED_FOXF1_TRACE_COMPARISON_PNG = QC_DIR / "fixed_foxf1_distance_trace_comparison.png"
FIXED_FOXF1_TRACE_FIXED_WIDTH_PNG = QC_DIR / "fixed_foxf1_distance_plot_fixed_width.png"
FIXED_FOXF1_TRACE_EQUAL_SUPPORT_PNG = QC_DIR / "fixed_foxf1_distance_plot_equal_support.png"
FOXF1_LIVE_VS_FIXED_COMPARISON_PNG = QC_DIR / "foxf1_live_vs_fixed_comparison.png"
FOXF1_LIVE_VS_FIXED_FIXED_WIDTH_PNG = QC_DIR / "foxf1_live_vs_fixed_fixed_width.png"
FOXF1_LIVE_VS_FIXED_EQUAL_SUPPORT_PNG = QC_DIR / "foxf1_live_vs_fixed_equal_support.png"
FIXED_SINGLE_CHANNEL_TRACE_GRID_PNG = QC_DIR / "fixed_single_channel_distance_plots.png"
FIXED_TAGYFP_TRACE_FIXED_WIDTH_PNG = QC_DIR / "fixed_foxf1_single_channel_fixed_width.png"
FIXED_TAGYFP_TRACE_EQUAL_SUPPORT_PNG = QC_DIR / "fixed_foxf1_single_channel_equal_support.png"
FIXED_SOX2_TRACE_FIXED_WIDTH_PNG = QC_DIR / "fixed_sox2_single_channel_fixed_width.png"
FIXED_SOX2_TRACE_EQUAL_SUPPORT_PNG = QC_DIR / "fixed_sox2_single_channel_equal_support.png"
FIXED_T_TRACE_FIXED_WIDTH_PNG = QC_DIR / "fixed_t_single_channel_fixed_width.png"
FIXED_T_TRACE_EQUAL_SUPPORT_PNG = QC_DIR / "fixed_t_single_channel_equal_support.png"
FIXED_FOXF1_SOX2_TRACE_FIXED_WIDTH_PNG = QC_DIR / "fixed_foxf1_sox2_distance_plot_fixed_width.png"
FIXED_FOXF1_SOX2_TRACE_EQUAL_SUPPORT_PNG = QC_DIR / "fixed_foxf1_sox2_distance_plot_equal_support.png"
FIXED_PAIRWISE_DENSITY_PNG = QC_DIR / "fixed_pairwise_density_plots.png"
FIXED_FOXF1_SOX2_DENSITY_PNG = QC_DIR / "fixed_foxf1_sox2_density.png"
FIXED_SOX2_T_DENSITY_PNG = QC_DIR / "fixed_sox2_t_density.png"
PAIRWISE_DENSITY_BINS = 160
PAIRWISE_DENSITY_SMOOTH_SIGMA = 1.2
PAIRWISE_DENSITY_LOWER_Q = 0.001
PAIRWISE_DENSITY_UPPER_Q = 0.999

## Fixed Quantification Strategy

This notebook is now fixed-only.

Current strategy:
1. Rebuild the accepted final fixed DAPI masks from notebook `04`.
2. Pool **raw DAPI pixels across all fixed images** and fit one global left-half Gaussian null.
3. Define a pixelwise DAPI gate at `mu_bg + 4 * sigma_bg`.
4. Keep only pixels that are both:
   - inside the final fixed DAPI mask
   - above the pooled raw DAPI gate
5. Apply any position-specific **channel-specific** manual pixel exclusions after the DAPI gate.
   - currently this can affect `FOXF1-YFP (fixed)`, `SOX2`, or `T`
   - exclusions are defined and reviewed per channel
6. For `FOXF1-YFP`, `SOX2`, and `T` separately:
   - pool **raw channel pixels across all fixed images**
   - fit one global left-half Gaussian null
   - convert each pixel to `(raw - mu_bg) / sigma_bg`
7. Compute pixelwise ratios:
   - `FOXF1-YFP_bgz / raw DAPI`
   - `SOX2_bgz / raw DAPI`
   - `T_bgz / raw DAPI`
8. Measure each ratio against distance from the nearest bead, exactly in fixed small-image space.
9. Plot all three channels against distance using:
   - fixed-width distance bins
   - equal-support bins
   - peak-normalized traces so `0` stays `0` and the peak trace bin is `1`

Important notes:
- there is **no flat-field correction** in this notebook
- the DAPI gate is **pixelwise only** and applies **no morphology**
- there is **no live TagYFP data or imagery** in this notebook

## Load Fixed Positions And Final Fixed DAPI Masks

In [ ]:
fixed_pos_df = common.filter_position_manifest(
    pos_df=common.load_position_manifest(POSITION_MANIFEST),
    cohort_ids=[FIXED_COHORT_ID],
    conditions=["fixed"],
).sort_values(["canonical_position"]).reset_index(drop=True)
fixed_pos_df["canonical_position"] = fixed_pos_df["canonical_position"].astype(str)
fixed_pos_by_cp = {str(r.canonical_position): pd.Series(r._asdict()) for r in fixed_pos_df.itertuples(index=False)}

fixed_cent_df = pd.read_csv(FIXED_SMALL_CENTROIDS, sep="	")
fixed_cent_df["canonical_position"] = fixed_cent_df["canonical_position"].astype(str)

final04 = lfq.build_final04_fixed_dapi_masks(
    position_manifest=POSITION_MANIFEST,
    cohort_id=FIXED_COHORT_ID,
    root=ROOT,
)
final04_summary_df = final04["summary_df"].copy()
final04_summary_df["canonical_position"] = final04_summary_df["canonical_position"].astype(str)
fixed_mask_payloads = final04["mask_payloads"]

print("Fixed positions:", len(fixed_pos_df))
print("Final04 fixed masks rebuilt:", len(fixed_mask_payloads))
display(final04_summary_df.head())

## Placeholder | Fixed Flat-Field Correction

No flat-field correction is currently applied in this notebook.

If we add fixed-channel flat-field correction later, this is where it should happen:
1. Estimate or load per-channel fixed flat-field models for `FOXF1-YFP`, `SOX2`, and `T`.
2. Apply those corrections to the raw fixed-channel images before pooled background fitting.
3. Then continue with the current global background standardization and DAPI normalization steps.

Current behavior remains unchanged: `FOXF1-YFP`, `SOX2`, and `T` use raw intensities, with no flat-field correction, before `(raw - mu_bg) / sigma_bg` standardization.

## Candidate Artifact Review

This early review uses **raw fixed-channel images** so we can inspect suspicious scenes before any background fitting or ratio construction.

Purpose:
- review candidate artifact scenes visually
- decide whether any channel-specific manual exclusion should exist at all
- keep this review logically upstream of the pooled fixed background models


In [ ]:
review_positions = []
flags_by_position = {}
for meas_name in MEASUREMENT_ORDER:
    for pos in CANDIDATE_ARTIFACT_REVIEW.get(meas_name, []):
        pos = str(pos)
        flags_by_position.setdefault(pos, []).append(MEASUREMENT_LABELS[meas_name].replace(" / DAPI", ""))
        if pos not in review_positions:
            review_positions.append(pos)


def _review_beads_for_position(pos: str) -> np.ndarray:
    cent_sub = fixed_cent_df[
        (fixed_cent_df["canonical_position"].astype(str) == str(pos))
        & (fixed_cent_df["mapping_status"] == "ok")
        & (fixed_cent_df["annotation_status"] == "annotated")
        & (fixed_cent_df["inside_fixed_small_fov"].fillna(False).astype(bool))
    ].copy()
    return cent_sub[["centroid_fixed_small_x_px", "centroid_fixed_small_y_px"]].to_numpy(dtype=np.float32)


def _review_mask_contour(ax, mask: np.ndarray, color: str = "lime", lw: float = 0.8) -> None:
    if np.any(mask):
        ax.contour(mask.astype(np.float32), levels=[0.5], colors=[color], linewidths=lw)


def _review_scatter_beads(ax, xy: np.ndarray, color: str = "magenta") -> None:
    if xy.size:
        ax.scatter(xy[:, 0], xy[:, 1], s=36, facecolors="none", edgecolors=color, linewidths=1.2)


raw_review_payloads = {}
for pos in review_positions:
    fixed_row = fixed_pos_by_cp[str(pos)]
    img = lfq.read_czi_with_optional_fixed_small_plane_selection(
        ROOT / str(fixed_row["primary_analysis_file"]),
        canonical_position=str(pos),
    )
    dapi_idx = common.find_channel_index(img.channels, ["dapi"])
    yfp_idx = common.find_channel_index(img.channels, ["tagyfp", "foxf1", "yfp"])
    sox2_idx = common.find_channel_index(img.channels, ["568", "alexa fluor 568"])
    t_idx = common.find_channel_index(img.channels, ["647", "alexa fluor 647"])
    raw_review_payloads[str(pos)] = {
        "final_mask": np.asarray(fixed_mask_payloads[str(pos)]["final_mask"], dtype=bool),
        "dapi_raw": np.asarray(img.channel_images[int(dapi_idx)], dtype=np.float32),
        "tagyfp_raw": np.asarray(img.channel_images[int(yfp_idx)], dtype=np.float32),
        "sox2_raw": np.asarray(img.channel_images[int(sox2_idx)], dtype=np.float32),
        "t_raw": np.asarray(img.channel_images[int(t_idx)], dtype=np.float32),
        "beads_xy_display": _review_beads_for_position(str(pos)),
    }

fig, axes = plt.subplots(len(review_positions), 4, figsize=(15.5, 3.8 * len(review_positions)), constrained_layout=True)
if len(review_positions) == 1:
    axes = np.asarray([axes])

for row_idx, pos in enumerate(review_positions):
    payload = raw_review_payloads[str(pos)]
    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    beads_xy = np.asarray(payload["beads_xy_display"], dtype=np.float32)
    flagged = ", ".join(flags_by_position.get(str(pos), []))

    ax = axes[row_idx, 0]
    ax.imshow(common._robust_rescale(payload["dapi_raw"]), cmap="gray")
    _review_mask_contour(ax, final_mask, color="lime")
    _review_scatter_beads(ax, beads_xy, color="magenta")
    ax.set_title(f"{pos} | Raw DAPI + final04 mask\nFlagged: {flagged}")
    ax.axis("off")

    ax = axes[row_idx, 1]
    ax.imshow(common._robust_rescale(payload["tagyfp_raw"]), cmap="gray")
    _review_mask_contour(ax, final_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | Raw FOXF1-YFP")
    ax.axis("off")

    ax = axes[row_idx, 2]
    ax.imshow(common._robust_rescale(payload["sox2_raw"]), cmap="gray")
    _review_mask_contour(ax, final_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | Raw SOX2")
    ax.axis("off")

    ax = axes[row_idx, 3]
    ax.imshow(common._robust_rescale(payload["t_raw"]), cmap="gray")
    _review_mask_contour(ax, final_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | Raw T")
    ax.axis("off")

fig.savefig(FIXED_CANDIDATE_ARTIFACT_REVIEW_PNG, dpi=180, bbox_inches="tight")
plt.show()

## Manual Pixel Exclusions

This is the approval step for the currently configured manual exclusions. These ROIs are defined **before** the pooled fixed background models and are reused later when the DAPI-gated channel-specific analysis masks are built.

Purpose:
- make the approved exclusion geometry explicit
- show the untouched raw channel before the ROI overlay
- keep the notebook story aligned with the actual computation order


In [ ]:
def _manual_prefit_exclusion_roi_preview(channel_key: str, pos: str, image_shape_yx: tuple[int, int]) -> tuple[np.ndarray, list[str]]:
    specs = MANUAL_CHANNEL_EXCLUSION_SPECS.get(str(channel_key), {}).get(str(pos), [])
    mask = np.zeros(image_shape_yx, dtype=bool)
    reasons: list[str] = []
    if not specs:
        return mask, reasons
    yy, xx = np.indices(image_shape_yx, dtype=np.float32)
    for spec in specs:
        shape = str(spec.get("shape", "ellipse"))
        reason = str(spec.get("reason", f"manual_{channel_key}_prefit_exclusion_{shape}"))
        if shape == "ellipse":
            cx = float(spec["cx"])
            cy = float(spec["cy"])
            rx = max(float(spec["rx"]), 1.0)
            ry = max(float(spec["ry"]), 1.0)
            part = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1.0
        elif shape == "rect":
            x0 = int(spec["x0"])
            x1 = int(spec["x1"])
            y0 = int(spec["y0"])
            y1 = int(spec["y1"])
            part = (xx >= x0) & (xx < x1) & (yy >= y0) & (yy < y1)
        else:
            raise ValueError(f"Unsupported manual exclusion shape for {channel_key} at {pos}: {shape}")
        mask |= np.asarray(part, dtype=bool)
        reasons.append(reason)
    return mask, reasons


approved_manual_entries = []
for channel_key in ["tagyfp", "sox2", "t"]:
    for pos in sorted(MANUAL_CHANNEL_EXCLUSION_SPECS.get(channel_key, {})):
        approved_manual_entries.append({"canonical_position": str(pos), "channel_key": str(channel_key)})

if approved_manual_entries:
    channel_labels = {"tagyfp": "FOXF1-YFP (fixed)", "sox2": "SOX2", "t": "T"}
    raw_keys = {"tagyfp": "tagyfp_raw", "sox2": "sox2_raw", "t": "t_raw"}

    fig, axes = plt.subplots(len(approved_manual_entries), 4, figsize=(16.5, 3.8 * len(approved_manual_entries)), constrained_layout=True)
    if len(approved_manual_entries) == 1:
        axes = np.asarray([axes])

    for row_idx, entry in enumerate(approved_manual_entries):
        pos = str(entry["canonical_position"])
        channel_key = str(entry["channel_key"])
        payload = raw_review_payloads.get(str(pos))
        if payload is None:
            fixed_row = fixed_pos_by_cp[str(pos)]
            img = lfq.read_czi_with_optional_fixed_small_plane_selection(
                ROOT / str(fixed_row["primary_analysis_file"]),
                canonical_position=str(pos),
            )
            dapi_idx = common.find_channel_index(img.channels, ["dapi"])
            yfp_idx = common.find_channel_index(img.channels, ["tagyfp", "foxf1", "yfp"])
            sox2_idx = common.find_channel_index(img.channels, ["568", "alexa fluor 568"])
            t_idx = common.find_channel_index(img.channels, ["647", "alexa fluor 647"])
            payload = {
                "final_mask": np.asarray(fixed_mask_payloads[str(pos)]["final_mask"], dtype=bool),
                "dapi_raw": np.asarray(img.channel_images[int(dapi_idx)], dtype=np.float32),
                "tagyfp_raw": np.asarray(img.channel_images[int(yfp_idx)], dtype=np.float32),
                "sox2_raw": np.asarray(img.channel_images[int(sox2_idx)], dtype=np.float32),
                "t_raw": np.asarray(img.channel_images[int(t_idx)], dtype=np.float32),
                "beads_xy_display": _review_beads_for_position(str(pos)),
            }
        final_mask = np.asarray(payload["final_mask"], dtype=bool)
        raw_arr = np.asarray(payload[raw_keys[channel_key]], dtype=np.float32)
        roi_mask, reasons = _manual_prefit_exclusion_roi_preview(channel_key, pos, raw_arr.shape)

        ax = axes[row_idx, 0]
        ax.imshow(common._robust_rescale(payload["dapi_raw"]), cmap="gray")
        _review_mask_contour(ax, final_mask, color="lime")
        _review_scatter_beads(ax, np.asarray(payload["beads_xy_display"], dtype=np.float32), color="magenta")
        ax.set_title(f"{pos} | Raw DAPI + final04 mask")
        ax.axis("off")

        ax = axes[row_idx, 1]
        ax.imshow(common._robust_rescale(raw_arr), cmap="gray")
        _review_mask_contour(ax, final_mask, color="white", lw=0.7)
        ax.set_title(f"{pos} | {channel_labels[channel_key]} raw")
        ax.axis("off")

        ax = axes[row_idx, 2]
        ax.imshow(common._robust_rescale(raw_arr), cmap="gray")
        overlay = np.zeros((*raw_arr.shape, 4), dtype=np.float32)
        overlay[roi_mask, 0] = 1.0
        overlay[roi_mask, 2] = 1.0
        overlay[roi_mask, 3] = 0.82
        ax.imshow(overlay)
        _review_mask_contour(ax, final_mask, color="white", lw=0.7)
        ax.set_title(f"{pos} | Approved exclusion ROI on raw channel")
        ax.axis("off")

        ax = axes[row_idx, 3]
        roi_show = np.where(roi_mask, raw_arr, np.nan)
        ax.imshow(roi_show, cmap="inferno")
        ax.set_title(f"{pos} | ROI values\nReasons: {'; '.join(reasons)}")
        ax.axis("off")

    fig.savefig(FIXED_MANUAL_EXCLUSION_APPROVAL_PNG, dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("No manual channel exclusions configured.")

## Global Fixed Background Models

In [ ]:
def _hist_weighted_quantile(hist: dict, q: float) -> float:
    counts = np.asarray(hist["counts"], dtype=np.float64)
    edges = np.asarray(hist["edges"], dtype=np.float64)
    q = float(np.clip(q, 0.0, 1.0))
    if counts.size == 0 or np.sum(counts) <= 0:
        return float(edges[-1])
    cdf = np.cumsum(counts)
    target = q * float(cdf[-1])
    idx = int(np.searchsorted(cdf, target, side="left"))
    idx = int(np.clip(idx, 0, len(edges) - 2))
    return float(edges[idx + 1])


def _manual_prefit_exclusion_roi(channel_key: str, pos: str, image_shape_yx: tuple[int, int]) -> tuple[np.ndarray, list[str]]:
    specs = MANUAL_CHANNEL_EXCLUSION_SPECS.get(str(channel_key), {}).get(str(pos), [])
    mask = np.zeros(image_shape_yx, dtype=bool)
    reasons: list[str] = []
    if not specs:
        return mask, reasons
    yy, xx = np.indices(image_shape_yx, dtype=np.float32)
    for spec in specs:
        shape = str(spec.get("shape", "ellipse"))
        reason = str(spec.get("reason", f"manual_{channel_key}_prefit_exclusion_{shape}"))
        if shape == "ellipse":
            cx = float(spec["cx"])
            cy = float(spec["cy"])
            rx = max(float(spec["rx"]), 1.0)
            ry = max(float(spec["ry"]), 1.0)
            part = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1.0
        elif shape == "rect":
            x0 = int(spec["x0"])
            x1 = int(spec["x1"])
            y0 = int(spec["y0"])
            y1 = int(spec["y1"])
            part = (xx >= x0) & (xx < x1) & (yy >= y0) & (yy < y1)
        else:
            raise ValueError(f"Unsupported manual prefit exclusion shape for {channel_key} at {pos}: {shape}")
        mask |= np.asarray(part, dtype=bool)
        reasons.append(reason)
    return mask, reasons


def pooled_histogram_fixed_channel_with_manual_exclusions(channel_spec: dict, n_bins: int = 2048, exclude_exact_zero_pixels: bool = True) -> dict:
    key = str(channel_spec["key"])
    keywords = list(channel_spec["keywords"])
    global_min = np.inf
    global_max = -np.inf
    n_pixels = 0
    n_images = 0
    integer_like = True
    manual_prefit_excluded_px = 0
    manual_prefit_positions: list[str] = []

    for pos in fixed_pos_df["canonical_position"].astype(str).tolist():
        fixed_row = fixed_pos_by_cp[str(pos)]
        img = lfq.read_czi_with_optional_fixed_small_plane_selection(
            ROOT / str(fixed_row["primary_analysis_file"]),
            canonical_position=str(pos),
        )
        ch_idx = common.find_channel_index(img.channels, keywords)
        if ch_idx is None:
            raise RuntimeError(f"Missing channel matching {keywords} in {fixed_row['primary_analysis_file']}")
        raw = img.channel_images[int(ch_idx)].astype(np.float32)
        exclusion_roi, _ = _manual_prefit_exclusion_roi(key, str(pos), raw.shape)
        valid = np.isfinite(raw) & (~exclusion_roi)
        if bool(exclude_exact_zero_pixels):
            valid &= raw != 0
        finite = raw[valid]
        excluded_here = np.isfinite(raw) & exclusion_roi
        if bool(exclude_exact_zero_pixels):
            excluded_here &= raw != 0
        excluded_count = int(np.sum(excluded_here))
        if excluded_count > 0:
            manual_prefit_excluded_px += excluded_count
            manual_prefit_positions.append(str(pos))
        if finite.size == 0:
            continue
        sample = finite if finite.size <= 4096 else finite[:: max(1, finite.size // 4096)]
        integer_like = bool(integer_like and np.all(np.abs(sample - np.rint(sample)) <= 1e-6))
        global_min = min(global_min, float(np.min(finite)))
        global_max = max(global_max, float(np.max(finite)))
        n_pixels += int(finite.size)
        n_images += 1

    if not np.isfinite(global_min) or not np.isfinite(global_max):
        raise RuntimeError(f"Could not determine finite histogram range for fixed channel {key}")
    if global_max <= global_min:
        global_max = global_min + 1.0

    if bool(integer_like):
        lo_i = int(np.floor(global_min))
        hi_i = int(np.ceil(global_max))
        edges = np.arange(lo_i - 0.5, hi_i + 1.5, 1.0, dtype=np.float64)
    else:
        edges = np.linspace(global_min, global_max, int(n_bins) + 1, dtype=np.float64)
    counts = np.zeros(len(edges) - 1, dtype=np.int64)

    for pos in fixed_pos_df["canonical_position"].astype(str).tolist():
        fixed_row = fixed_pos_by_cp[str(pos)]
        img = lfq.read_czi_with_optional_fixed_small_plane_selection(
            ROOT / str(fixed_row["primary_analysis_file"]),
            canonical_position=str(pos),
        )
        ch_idx = common.find_channel_index(img.channels, keywords)
        if ch_idx is None:
            raise RuntimeError(f"Missing channel matching {keywords} in {fixed_row['primary_analysis_file']}")
        raw = img.channel_images[int(ch_idx)].astype(np.float32)
        exclusion_roi, _ = _manual_prefit_exclusion_roi(key, str(pos), raw.shape)
        valid = np.isfinite(raw) & (~exclusion_roi)
        if bool(exclude_exact_zero_pixels):
            valid &= raw != 0
        finite = raw[valid]
        if finite.size == 0:
            continue
        h, _ = np.histogram(finite, bins=edges)
        counts += h.astype(np.int64)

    centers = 0.5 * (edges[:-1] + edges[1:])
    return {
        "edges": edges,
        "centers": centers,
        "counts": counts,
        "global_min": float(global_min),
        "global_max": float(global_max),
        "n_pixels": int(n_pixels),
        "n_images": int(n_images),
        "n_bins": int(len(counts)),
        "integer_like_bins": bool(integer_like),
        "manual_prefit_excluded_px": int(manual_prefit_excluded_px),
        "manual_prefit_positions": sorted(set(manual_prefit_positions)),
    }


def fit_pooled_channel_background(channel_spec: dict) -> dict:
    key = str(channel_spec["key"])
    if MANUAL_CHANNEL_EXCLUSION_SPECS.get(key):
        hist = pooled_histogram_fixed_channel_with_manual_exclusions(
            channel_spec=channel_spec,
            n_bins=2048,
            exclude_exact_zero_pixels=True,
        )
    else:
        hist = lfq.pooled_histogram_manifest_channel(
            position_manifest=POSITION_MANIFEST,
            cohort_ids=[FIXED_COHORT_ID],
            conditions=["fixed"],
            channel_keywords=channel_spec["keywords"],
            n_bins=2048,
            exclude_exact_zero_pixels=True,
        )
        hist["manual_prefit_excluded_px"] = 0
        hist["manual_prefit_positions"] = []
    fit = common.estimate_background_half_gaussian(
        hist=hist,
        smooth_sigma_bins=float(HIST_SMOOTH_SIGMA_BINS),
    )
    return {"hist": hist, "fit": fit}

bg_models = {spec["key"]: fit_pooled_channel_background(spec) for spec in CHANNEL_SPECS}

bg_rows = []
for spec in CHANNEL_SPECS:
    key = spec["key"]
    payload = bg_models[key]
    fit = payload["fit"]
    mu_bg = float(fit["mu_bg_raw"])
    sigma_bg = float(fit["sigma_bg_raw"])
    row = {
        "channel_key": key,
        "channel_label": spec["label"],
        "keywords": ";".join(spec["keywords"]),
        "mu_bg_raw": mu_bg,
        "sigma_bg_raw": sigma_bg,
        "n_pixels": int(fit["n_pixels"]),
        "n_images": int(fit["n_images"]),
        "fit_method": str(fit["fit_method"]),
        "manual_prefit_excluded_px": int(payload["hist"].get("manual_prefit_excluded_px", 0)),
        "manual_prefit_exclusion_positions": ";".join(payload["hist"].get("manual_prefit_positions", [])),
        "reference_n_sigma": float(DAPI_GATE_N_SIGMA),
        "mu_plus_nsigma": float(mu_bg + DAPI_GATE_N_SIGMA * sigma_bg),
    }
    if key == "dapi":
        row["analysis_gate_threshold"] = float(mu_bg + DAPI_GATE_N_SIGMA * sigma_bg)
    bg_rows.append(row)

fixed_global_bg_df = pd.DataFrame(bg_rows)
fixed_global_bg_df.to_csv(FIXED_GLOBAL_BG_TSV, sep="	", index=False)

display(fixed_global_bg_df)

fig, axes = plt.subplots(2, 4, figsize=(19.0, 9.0), constrained_layout=True)
for col_idx, spec in enumerate(CHANNEL_SPECS):
    key = spec["key"]
    payload = bg_models[key]
    hist = payload["hist"]
    fit = payload["fit"]
    x = np.asarray(hist["centers"], dtype=float)
    counts = np.asarray(hist["counts"], dtype=float)
    edges = np.asarray(hist["edges"], dtype=float)
    widths = np.diff(edges)
    smooth = np.asarray(fit["smooth_counts"], dtype=float)
    fit_counts = np.asarray(fit["fit_counts"], dtype=float)
    mu_bg = float(fit["mu_bg_raw"])
    sigma_bg = float(fit["sigma_bg_raw"])
    x_q99 = _hist_weighted_quantile(hist, FULL_HIST_UPPER_QUANTILE)
    nsigma_x = float(mu_bg + DAPI_GATE_N_SIGMA * sigma_bg)

    ax = axes[0, col_idx]
    ax.bar(x, counts, width=widths, align="center", color="0.86", edgecolor="0.72", linewidth=0.2, label="Pooled histogram")
    ax.plot(x, smooth, color="0.35", lw=1.5, label="Smoothed histogram")
    ax.plot(x, fit_counts, color="crimson", lw=2.0, label="Left-half Gaussian fit")
    ax.axvline(mu_bg, color="black", lw=1.2, ls="--", label="mu_bg")
    if key == "dapi":
        ax.axvline(nsigma_x, color="magenta", lw=1.5, ls=":", label=f"analysis gate cutoff ({DAPI_GATE_N_SIGMA:.0f} sigma)")
    else:
        ax.axvline(nsigma_x, color="royalblue", lw=1.2, ls=":", label=f"mu_bg + {DAPI_GATE_N_SIGMA:.0f} sigma_bg")
    ax.set_xlim(float(edges[0]), float(x_q99))
    ax.set_title(f"{spec['label']} | full distribution (to 99th percentile)")
    ax.set_xlabel("Raw intensity")
    ax.set_ylabel("Pixel count")
    ax.text(
        0.98,
        0.95,
        f"mu = {mu_bg:.2f}\nsigma = {sigma_bg:.2f}\nN = {int(fit['n_pixels']):,}\nmanual excl = {int(hist.get('manual_prefit_excluded_px', 0)):,}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.95},
    )

    ax = axes[1, col_idx]
    ax.bar(x, counts, width=widths, align="center", color="0.86", edgecolor="0.72", linewidth=0.2)
    ax.plot(x, smooth, color="0.35", lw=1.5)
    ax.plot(x, fit_counts, color="crimson", lw=2.0)
    ax.axvline(mu_bg, color="black", lw=1.2, ls="--")
    x_lo = max(float(edges[0]), float(mu_bg - NULL_PEAK_LEFT_SIGMA * sigma_bg))
    x_hi = min(float(x_q99), float(mu_bg + NULL_PEAK_RIGHT_SIGMA * sigma_bg))
    if x_hi <= x_lo:
        x_lo = float(max(edges[0], mu_bg - 3.0 * sigma_bg))
        x_hi = float(min(x_q99, mu_bg + 3.0 * sigma_bg))
    ax.set_xlim(x_lo, x_hi)
    ax.set_title(f"{spec['label']} | null peak zoom")
    ax.set_xlabel("Raw intensity")
    ax.set_ylabel("Pixel count")

axes[0, 0].legend(loc="upper left", fontsize=8)
fig.savefig(FIXED_GLOBAL_BG_FITS_PNG, dpi=180, bbox_inches="tight")
plt.show()

## DAPI Gate Summary

In [ ]:
def _mask_contour(ax, mask: np.ndarray, color: str = "cyan", lw: float = 0.8) -> None:
    if np.any(mask):
        ax.contour(mask.astype(np.float32), levels=[0.5], colors=[color], linewidths=lw)


def _scatter_beads(ax, xy: np.ndarray, color: str = "magenta") -> None:
    if xy.size:
        ax.scatter(
            xy[:, 0],
            xy[:, 1],
            s=36,
            facecolors="none",
            edgecolors=color,
            linewidths=1.2,
        )


def aggregate_trace_across_images(trace_df: pd.DataFrame) -> pd.DataFrame:
    if len(trace_df) == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um", "mean", "sd", "sem", "n_images"])
    rows = []
    group_cols = ["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um"]
    for keys, sub in trace_df.groupby(group_cols, sort=True):
        vals = sub["mean_value"].to_numpy(dtype=float)
        n = int(sub["canonical_position"].astype(str).nunique())
        sd = float(np.nanstd(vals, ddof=1)) if n > 1 else 0.0
        sem = float(sd / np.sqrt(n)) if n > 1 else 0.0
        rows.append(
            {
                "measurement_name": str(keys[0]),
                "bin_idx": int(keys[1]),
                "bin_start_um": float(keys[2]),
                "bin_end_um": float(keys[3]),
                "bin_mid_um": float(keys[4]),
                "mean": float(np.nanmean(vals)),
                "sd": sd,
                "sem": sem,
                "n_images": n,
            }
        )
    return pd.DataFrame(rows).sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)


def weighted_trace_pixels(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame(columns=[
            "measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um",
            "total_count_px", "weighted_mean_value", "pooled_sd_value", "pooled_sem_value"
        ])
    tmp = df.copy()
    counts = tmp["count_px"].astype(float)
    means = tmp["mean_value"].astype(float)
    stds = pd.to_numeric(tmp["std_value"], errors="coerce").astype(float)
    within_ss = np.where((counts > 1) & np.isfinite(stds), np.square(stds) * np.maximum(counts - 1.0, 0.0), 0.0)
    tmp["weighted_sum"] = means * counts
    tmp["sum_x2"] = within_ss + counts * np.square(means)
    agg = (
        tmp.groupby(["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um"], as_index=False)
        .agg(
            total_count_px=("count_px", "sum"),
            total_weighted_sum=("weighted_sum", "sum"),
            total_sum_x2=("sum_x2", "sum"),
        )
    )
    total_n = np.maximum(agg["total_count_px"].astype(float), 1.0)
    agg["weighted_mean_value"] = agg["total_weighted_sum"] / total_n
    numer = agg["total_sum_x2"] - np.square(agg["total_weighted_sum"]) / total_n
    denom = np.maximum(total_n - 1.0, 1.0)
    agg["pooled_sd_value"] = np.sqrt(np.maximum(numer / denom, 0.0))
    agg.loc[agg["total_count_px"].astype(float) <= 1.0, "pooled_sd_value"] = 0.0
    agg["pooled_sem_value"] = agg["pooled_sd_value"] / np.sqrt(total_n)
    return agg.sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)


def equal_support_trace(distance_um: np.ndarray, values: np.ndarray, pixels_per_bin: int, measurement_name: str) -> pd.DataFrame:
    dist = np.asarray(distance_um, dtype=np.float32)
    vals = np.asarray(values, dtype=np.float32)
    keep = np.isfinite(dist) & np.isfinite(vals)
    dist = dist[keep]
    vals = vals[keep]
    if dist.size == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um", "count_px", "mean_value", "median_value", "std_value", "sem_value"])
    order = np.argsort(dist, kind="mergesort")
    dist = dist[order]
    vals = vals[order]
    rows = []
    pixels_per_bin = max(1, int(pixels_per_bin))
    for start in range(0, len(dist), pixels_per_bin):
        stop = min(len(dist), start + pixels_per_bin)
        d = dist[start:stop]
        v = vals[start:stop]
        sd = float(np.nanstd(v, ddof=1)) if len(v) > 1 else 0.0
        sem = float(sd / np.sqrt(len(v))) if len(v) > 1 else 0.0
        rows.append(
            {
                "measurement_name": str(measurement_name),
                "bin_idx": int(len(rows)),
                "bin_start_um": float(d[0]),
                "bin_end_um": float(d[-1]),
                "bin_mid_um": float(np.nanmean(d)),
                "count_px": int(len(v)),
                "mean_value": float(np.nanmean(v)),
                "median_value": float(np.nanmedian(v)),
                "std_value": sd,
                "sem_value": sem,
            }
        )
    return pd.DataFrame(rows)


def aggregate_equal_support_across_images(payloads: dict, equal_support_df: pd.DataFrame, measurement_name: str) -> pd.DataFrame:
    rows = []
    for _, rr in equal_support_df.sort_values("bin_idx").iterrows():
        lo = float(rr["bin_start_um"])
        hi = float(rr["bin_end_um"])
        image_means = []
        image_counts = []
        for pos, payload in payloads.items():
            dist = np.asarray(payload["distance_um"], dtype=np.float32)
            vals = np.asarray(payload["value"], dtype=np.float32)
            keep = np.isfinite(dist) & np.isfinite(vals) & (dist >= lo - 1e-9) & (dist <= hi + 1e-9)
            if not np.any(keep):
                continue
            vv = vals[keep]
            image_means.append(float(np.nanmean(vv)))
            image_counts.append(int(vv.size))
        arr = np.asarray(image_means, dtype=float)
        n = int(np.sum(np.isfinite(arr)))
        mean = float(np.nanmean(arr)) if n else np.nan
        sd = float(np.nanstd(arr, ddof=1)) if n > 1 else (0.0 if n == 1 else np.nan)
        sem = float(sd / np.sqrt(n)) if n > 1 else (0.0 if n == 1 else np.nan)
        rows.append(
            {
                "measurement_name": str(measurement_name),
                "bin_idx": int(rr["bin_idx"]),
                "bin_start_um": lo,
                "bin_end_um": hi,
                "bin_mid_um": float(rr["bin_mid_um"]),
                "bin_width_um": float(hi - lo),
                "total_count_px": int(np.sum(image_counts)) if image_counts else 0,
                "n_images": n,
                "mean": mean,
                "sd": sd,
                "sem": sem,
            }
        )
    return pd.DataFrame(rows).sort_values("bin_idx").reset_index(drop=True)


def _channel_bg_standardize(raw: np.ndarray, mu_bg: float, sigma_bg: float) -> np.ndarray:
    return ((np.asarray(raw, dtype=np.float32) - float(mu_bg)) / max(float(sigma_bg), float(lfq.EPS))).astype(np.float32)


def _beads_for_position(pos: str) -> tuple[np.ndarray, np.ndarray]:
    cent_sub = fixed_cent_df[
        (fixed_cent_df["canonical_position"].astype(str) == str(pos))
        & (fixed_cent_df["mapping_status"] == "ok")
        & (fixed_cent_df["annotation_status"] == "annotated")
    ].copy()
    all_xy = cent_sub[["centroid_fixed_small_x_px", "centroid_fixed_small_y_px"]].to_numpy(dtype=np.float32)
    disp_sub = cent_sub[cent_sub["inside_fixed_small_fov"].fillna(False).astype(bool)].copy()
    display_xy = disp_sub[["centroid_fixed_small_x_px", "centroid_fixed_small_y_px"]].to_numpy(dtype=np.float32)
    return all_xy, display_xy


def _manual_channel_exclusion_mask(channel_key: str, pos: str, image_shape_yx: tuple[int, int]) -> tuple[np.ndarray, list[str]]:
    specs = MANUAL_CHANNEL_EXCLUSION_SPECS.get(str(channel_key), {}).get(str(pos), [])
    mask = np.zeros(image_shape_yx, dtype=bool)
    reasons: list[str] = []
    if not specs:
        return mask, reasons
    yy, xx = np.indices(image_shape_yx, dtype=np.float32)
    for spec in specs:
        shape = str(spec.get("shape", "ellipse"))
        reason = str(spec.get("reason", f"manual_{channel_key}_exclusion_{shape}"))
        if shape == "ellipse":
            cx = float(spec["cx"])
            cy = float(spec["cy"])
            rx = max(float(spec["rx"]), 1.0)
            ry = max(float(spec["ry"]), 1.0)
            part = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1.0
        elif shape == "rect":
            x0 = int(spec["x0"])
            x1 = int(spec["x1"])
            y0 = int(spec["y0"])
            y1 = int(spec["y1"])
            part = (xx >= x0) & (xx < x1) & (yy >= y0) & (yy < y1)
        else:
            raise ValueError(f"Unsupported manual exclusion shape for {channel_key} at {pos}: {shape}")
        mask |= np.asarray(part, dtype=bool)
        reasons.append(reason)
    return mask, reasons


def _plot_multi_channel_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, title: str, ylabel: str) -> None:
    for meas_name in MEASUREMENT_ORDER:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        per_sub = per_image_df[per_image_df["measurement_name"] == meas_name].copy()
        agg_sub = agg_df[agg_df["measurement_name"] == meas_name].copy()
        for _, sub in per_sub.groupby("canonical_position", sort=True):
            sub = sub.sort_values("bin_mid_um")
            ax.plot(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.10, linewidth=0.8)
            ax.scatter(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.06, s=6)
        if len(agg_sub) > 0:
            agg_sub = agg_sub.sort_values("bin_mid_um")
            mean_vals = agg_sub["mean"].to_numpy(dtype=float)
            sd_vals = agg_sub["sd"].to_numpy(dtype=float)
            ax.plot(agg_sub["bin_mid_um"], mean_vals, color=color, linewidth=2.4, label=label)
            ax.fill_between(agg_sub["bin_mid_um"], mean_vals - sd_vals, mean_vals + sd_vals, color=color, alpha=0.18)
    ax.set_xlabel("Distance from nearest bead (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best")


def _plot_multi_channel_merged(ax, merged_df: pd.DataFrame, title: str, ylabel: str, mean_col: str, sd_col: str, count_col: str | None = None, add_ci95: bool = True) -> None:
    for meas_name in MEASUREMENT_ORDER:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        sub = merged_df[merged_df["measurement_name"] == meas_name].copy().sort_values("bin_mid_um")
        if len(sub) == 0:
            continue
        mean_vals = sub[mean_col].to_numpy(dtype=float)
        sd_vals = sub[sd_col].to_numpy(dtype=float)
        ax.plot(sub["bin_mid_um"], mean_vals, color=color, linewidth=2.3, label=label)
        ax.fill_between(sub["bin_mid_um"], mean_vals - sd_vals, mean_vals + sd_vals, color=color, alpha=0.16)
        if add_ci95 and count_col is not None and count_col in sub.columns:
            n = np.maximum(sub[count_col].to_numpy(dtype=float), 1.0)
            ci95 = 1.96 * sd_vals / np.sqrt(n)
            ax.plot(sub["bin_mid_um"], mean_vals - ci95, color=color, lw=1.2, ls="--", alpha=0.9)
            ax.plot(sub["bin_mid_um"], mean_vals + ci95, color=color, lw=1.2, ls="--", alpha=0.9)
    ax.set_xlabel("Distance from nearest bead (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="upper left")


dapi_gate_threshold = float(fixed_global_bg_df.loc[fixed_global_bg_df["channel_key"] == "dapi", "analysis_gate_threshold"].iloc[0])
bg_mu_map = fixed_global_bg_df.set_index("channel_key")["mu_bg_raw"].astype(float).to_dict()
bg_sigma_map = fixed_global_bg_df.set_index("channel_key")["sigma_bg_raw"].astype(float).to_dict()

analysis_payloads = {}
gate_rows = []
ratio_rows = []
pixel_payloads_by_measurement = {meas: {} for meas in MEASUREMENT_ORDER}
measurement_position_exclusions = {str(k): set(map(str, v)) for k, v in MANUAL_MEASUREMENT_POSITION_EXCLUSIONS.items()}

for pos in fixed_pos_df["canonical_position"].astype(str).tolist():
    payload = fixed_mask_payloads[str(pos)]
    fixed_row = fixed_pos_by_cp[str(pos)]
    img = lfq.read_czi_with_optional_fixed_small_plane_selection(
        ROOT / str(fixed_row["primary_analysis_file"]),
        canonical_position=str(pos),
    )
    bf_idx = common.find_channel_index(img.channels, ["bright"])
    dapi_idx = common.find_channel_index(img.channels, ["dapi"])
    yfp_idx = common.find_channel_index(img.channels, ["tagyfp", "foxf1", "yfp"])
    sox2_idx = common.find_channel_index(img.channels, ["568", "alexa fluor 568"])
    t_idx = common.find_channel_index(img.channels, ["647", "alexa fluor 647"])
    needed = [bf_idx, dapi_idx, yfp_idx, sox2_idx, t_idx]
    if any(v is None for v in needed):
        raise RuntimeError(f"Missing required fixed channels in {fixed_row['primary_analysis_file']}; channels={img.channels}")

    bf = np.asarray(img.channel_images[int(bf_idx)], dtype=np.float32)
    dapi_raw = np.asarray(img.channel_images[int(dapi_idx)], dtype=np.float32)
    tagyfp_raw = np.asarray(img.channel_images[int(yfp_idx)], dtype=np.float32)
    sox2_raw = np.asarray(img.channel_images[int(sox2_idx)], dtype=np.float32)
    t_raw = np.asarray(img.channel_images[int(t_idx)], dtype=np.float32)

    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    finite_all = np.isfinite(dapi_raw) & np.isfinite(tagyfp_raw) & np.isfinite(sox2_raw) & np.isfinite(t_raw)
    dapi_gate_keep = finite_all & (dapi_raw >= dapi_gate_threshold)
    analysis_mask = final_mask & dapi_gate_keep
    removed_by_dapi_gate = final_mask & (~dapi_gate_keep)

    channel_manual_exclusion_masks = {}
    channel_analysis_masks = {}
    channel_manual_reasons = {}
    for channel_key in ["tagyfp", "sox2", "t"]:
        exclusion_roi, exclusion_reasons = _manual_channel_exclusion_mask(channel_key, str(pos), dapi_raw.shape)
        exclusion_mask = np.asarray(analysis_mask & exclusion_roi, dtype=bool)
        channel_manual_exclusion_masks[channel_key] = exclusion_mask
        channel_analysis_masks[channel_key] = np.asarray(analysis_mask & (~exclusion_mask), dtype=bool)
        channel_manual_reasons[channel_key] = list(exclusion_reasons)

    tagyfp_manual_exclusion_mask = channel_manual_exclusion_masks["tagyfp"]
    sox2_manual_exclusion_mask = channel_manual_exclusion_masks["sox2"]
    t_manual_exclusion_mask = channel_manual_exclusion_masks["t"]
    tagyfp_analysis_mask = channel_analysis_masks["tagyfp"]
    sox2_analysis_mask = channel_analysis_masks["sox2"]
    t_analysis_mask = channel_analysis_masks["t"]
    foxf1_manual_exclusion_reasons = list(channel_manual_reasons["tagyfp"])
    foxf1_analysis_mask = tagyfp_analysis_mask

    tagyfp_bgz = _channel_bg_standardize(tagyfp_raw, bg_mu_map["tagyfp"], bg_sigma_map["tagyfp"])
    sox2_bgz = _channel_bg_standardize(sox2_raw, bg_mu_map["sox2"], bg_sigma_map["sox2"])
    t_bgz = _channel_bg_standardize(t_raw, bg_mu_map["t"], bg_sigma_map["t"])

    with np.errstate(divide="ignore", invalid="ignore"):
        tagyfp_ratio = np.asarray(tagyfp_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)
        sox2_ratio = np.asarray(sox2_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)
        t_ratio = np.asarray(t_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)

    beads_xy_all, beads_xy_display = _beads_for_position(str(pos))
    if beads_xy_all.size == 0:
        raise RuntimeError(f"Missing mapped fixed bead centroids for {pos}")
    dist_um = common.nearest_bead_distance_map_um(
        image_shape_yx=dapi_raw.shape,
        centroid_xy_px=beads_xy_all,
        pixel_um_x=float(img.pixel_um_x),
        pixel_um_y=float(img.pixel_um_y),
    )

    analysis_payloads[str(pos)] = {
        "image_id": str(payload["row"].get("image_id", f"{FIXED_COHORT_ID}_{pos}")),
        "bf": bf,
        "dapi_raw": dapi_raw,
        "tagyfp_raw": tagyfp_raw,
        "sox2_raw": sox2_raw,
        "t_raw": t_raw,
        "final_mask": final_mask,
        "removed_by_dapi_gate": removed_by_dapi_gate,
        "analysis_mask": analysis_mask,
        "tagyfp_analysis_mask": tagyfp_analysis_mask,
        "sox2_analysis_mask": sox2_analysis_mask,
        "t_analysis_mask": t_analysis_mask,
        "tagyfp_manual_exclusion_mask": tagyfp_manual_exclusion_mask,
        "sox2_manual_exclusion_mask": sox2_manual_exclusion_mask,
        "t_manual_exclusion_mask": t_manual_exclusion_mask,
        "tagyfp_manual_exclusion_reasons": list(channel_manual_reasons["tagyfp"]),
        "sox2_manual_exclusion_reasons": list(channel_manual_reasons["sox2"]),
        "t_manual_exclusion_reasons": list(channel_manual_reasons["t"]),
        "foxf1_analysis_mask": tagyfp_analysis_mask,
        "foxf1_manual_exclusion_mask": tagyfp_manual_exclusion_mask,
        "foxf1_manual_exclusion_reasons": list(channel_manual_reasons["tagyfp"]),
        "dapi_gate_keep": dapi_gate_keep,
        "tagyfp_bgz": tagyfp_bgz,
        "sox2_bgz": sox2_bgz,
        "t_bgz": t_bgz,
        "tagyfp_ratio": tagyfp_ratio,
        "sox2_ratio": sox2_ratio,
        "t_ratio": t_ratio,
        "distance_um": dist_um,
        "beads_xy_all": beads_xy_all,
        "beads_xy_display": beads_xy_display,
    }

    gate_rows.append(
        {
            "canonical_position": str(pos),
            "image_id": str(payload["row"].get("image_id", f"{FIXED_COHORT_ID}_{pos}")),
            "final_mask_area_px": int(np.sum(final_mask)),
            "analysis_mask_area_px": int(np.sum(analysis_mask)),
            "removed_by_dapi_gate_px": int(np.sum(removed_by_dapi_gate)),
            "tagyfp_analysis_mask_area_px": int(np.sum(tagyfp_analysis_mask)),
            "sox2_analysis_mask_area_px": int(np.sum(sox2_analysis_mask)),
            "t_analysis_mask_area_px": int(np.sum(t_analysis_mask)),
            "tagyfp_manual_exclusion_px": int(np.sum(tagyfp_manual_exclusion_mask)),
            "sox2_manual_exclusion_px": int(np.sum(sox2_manual_exclusion_mask)),
            "t_manual_exclusion_px": int(np.sum(t_manual_exclusion_mask)),
            "tagyfp_manual_exclusion_fraction_within_analysis_mask": float(np.sum(tagyfp_manual_exclusion_mask) / max(np.sum(analysis_mask), 1)),
            "sox2_manual_exclusion_fraction_within_analysis_mask": float(np.sum(sox2_manual_exclusion_mask) / max(np.sum(analysis_mask), 1)),
            "t_manual_exclusion_fraction_within_analysis_mask": float(np.sum(t_manual_exclusion_mask) / max(np.sum(analysis_mask), 1)),
            "retained_fraction_within_mask": float(np.sum(analysis_mask) / max(np.sum(final_mask), 1)),
            "removed_fraction_within_mask": float(np.sum(removed_by_dapi_gate) / max(np.sum(final_mask), 1)),
            "dapi_gate_threshold": float(dapi_gate_threshold),
            "tagyfp_manual_exclusion_reasons": ";".join(channel_manual_reasons["tagyfp"]),
            "sox2_manual_exclusion_reasons": ";".join(channel_manual_reasons["sox2"]),
            "t_manual_exclusion_reasons": ";".join(channel_manual_reasons["t"]),
            "foxf1_analysis_mask_area_px": int(np.sum(tagyfp_analysis_mask)),
            "foxf1_manual_exclusion_px": int(np.sum(tagyfp_manual_exclusion_mask)),
            "foxf1_manual_exclusion_fraction_within_analysis_mask": float(np.sum(tagyfp_manual_exclusion_mask) / max(np.sum(analysis_mask), 1)),
            "foxf1_manual_exclusion_reasons": ";".join(channel_manual_reasons["tagyfp"]),
            "n_beads_total": int(len(beads_xy_all)),
            "n_beads_displayed": int(len(beads_xy_display)),
        }
    )

    signal_map = {
        "fixed_tagyfp_bgz_over_dapi_gate": tagyfp_ratio,
        "fixed_sox2_bgz_over_dapi_gate": sox2_ratio,
        "fixed_t_bgz_over_dapi_gate": t_ratio,
    }
    measurement_masks = {
        "fixed_tagyfp_bgz_over_dapi_gate": tagyfp_analysis_mask,
        "fixed_sox2_bgz_over_dapi_gate": sox2_analysis_mask,
        "fixed_t_bgz_over_dapi_gate": t_analysis_mask,
    }
    measurement_mask_sources = {
        "fixed_tagyfp_bgz_over_dapi_gate": "final04_fixed_mask_plus_pooled_raw_dapi_gate_plus_channel_specific_manual_exclusion_if_any",
        "fixed_sox2_bgz_over_dapi_gate": "final04_fixed_mask_plus_pooled_raw_dapi_gate_plus_channel_specific_manual_exclusion_if_any",
        "fixed_t_bgz_over_dapi_gate": "final04_fixed_mask_plus_pooled_raw_dapi_gate_plus_channel_specific_manual_exclusion_if_any",
    }
    for meas_name, arr in signal_map.items():
        if str(pos) in measurement_position_exclusions.get(str(meas_name), set()):
            continue
        meas_mask = np.asarray(measurement_masks[meas_name], dtype=bool)
        stats = lfq.masked_distance_bin_stats(
            value_img=arr,
            distance_um_map=dist_um,
            mask=meas_mask,
            bin_um=float(DIST_BIN_UM),
        )
        for _, tr in stats.iterrows():
            ratio_rows.append(
                {
                    "canonical_position": str(pos),
                    "image_id": str(payload["row"].get("image_id", f"{FIXED_COHORT_ID}_{pos}")),
                    "cohort_id": FIXED_COHORT_ID,
                    "small_file_path": str(fixed_row["primary_analysis_file"]),
                    "measurement_name": str(meas_name),
                    "mask_source": measurement_mask_sources[meas_name],
                    "bead_distance_mode": "nearest",
                    "bin_idx": int(tr["bin_idx"]),
                    "bin_start_um": float(tr["bin_start_um"]),
                    "bin_end_um": float(tr["bin_end_um"]),
                    "bin_mid_um": float(tr["bin_mid_um"]),
                    "count_px": int(tr["count_px"]),
                    "mean_value": float(tr["mean_value"]),
                    "median_value": float(tr["median_value"]),
                    "std_value": float(tr["std_value"]) if pd.notna(tr["std_value"]) else np.nan,
                    "sem_value": float(tr["sem_value"]) if pd.notna(tr["sem_value"]) else np.nan,
                }
            )
        keep = meas_mask & np.isfinite(arr) & np.isfinite(dist_um)
        pixel_payloads_by_measurement[meas_name][str(pos)] = {
            "distance_um": np.asarray(dist_um[keep], dtype=np.float32),
            "value": np.asarray(arr[keep], dtype=np.float32),
        }

fixed_dapi_gate_summary_df = pd.DataFrame(gate_rows).sort_values("canonical_position").reset_index(drop=True)
fixed_dapi_gate_summary_df.to_csv(FIXED_DAPI_GATE_SUMMARY_TSV, sep="	", index=False)

print("DAPI analysis gate threshold:", f"{dapi_gate_threshold:.3f}")
print("Median retained fraction within final mask:", f"{fixed_dapi_gate_summary_df['retained_fraction_within_mask'].median():.3f}")
print("Positions with largest gate removals:")
display(fixed_dapi_gate_summary_df.sort_values("removed_fraction_within_mask", ascending=False).head(10))

## DAPI Gate Debugging

### DAPI Gate Sigma Sweep

Representative positions are shown across a wide range of pooled raw-DAPI cutoffs (`mu_bg + N * sigma_bg`) to bracket the transition from essentially no pixel removal to visible trimming of real DAPI-positive pixels.

In [ ]:
gate_sweep_rows = []
for pos in DAPI_GATE_SWEEP_POSITIONS:
    payload = analysis_payloads[str(pos)]
    dapi_raw = np.asarray(payload["dapi_raw"], dtype=np.float32)
    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    finite_all = np.isfinite(dapi_raw)
    for n_sigma in DAPI_GATE_SWEEP_N_SIGMA:
        threshold = float(bg_mu_map["dapi"] + float(n_sigma) * bg_sigma_map["dapi"])
        keep_mask = finite_all & (dapi_raw >= threshold)
        analysis_mask_sweep = final_mask & keep_mask
        removed_sweep = final_mask & (~keep_mask)
        gate_sweep_rows.append(
            {
                "canonical_position": str(pos),
                "n_sigma": float(n_sigma),
                "threshold": threshold,
                "final_mask_area_px": int(np.sum(final_mask)),
                "analysis_mask_area_px": int(np.sum(analysis_mask_sweep)),
                "removed_by_gate_px": int(np.sum(removed_sweep)),
                "removed_fraction_within_mask": float(np.sum(removed_sweep) / max(np.sum(final_mask), 1)),
            }
        )

gate_sweep_df = pd.DataFrame(gate_sweep_rows)
display(gate_sweep_df)

fig, axes = plt.subplots(
    len(DAPI_GATE_SWEEP_POSITIONS),
    len(DAPI_GATE_SWEEP_N_SIGMA),
    figsize=(3.2 * len(DAPI_GATE_SWEEP_N_SIGMA), 3.8 * len(DAPI_GATE_SWEEP_POSITIONS)),
    constrained_layout=True,
)
if len(DAPI_GATE_SWEEP_POSITIONS) == 1:
    axes = np.asarray([axes])
if len(DAPI_GATE_SWEEP_N_SIGMA) == 1:
    axes = axes[:, np.newaxis]

for row_idx, pos in enumerate(DAPI_GATE_SWEEP_POSITIONS):
    payload = analysis_payloads[str(pos)]
    dapi_raw = np.asarray(payload["dapi_raw"], dtype=np.float32)
    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    finite_all = np.isfinite(dapi_raw)
    beads_xy = payload["beads_xy_display"]
    for col_idx, n_sigma in enumerate(DAPI_GATE_SWEEP_N_SIGMA):
        threshold = float(bg_mu_map["dapi"] + float(n_sigma) * bg_sigma_map["dapi"])
        keep_mask = finite_all & (dapi_raw >= threshold)
        analysis_mask_sweep = final_mask & keep_mask
        removed_sweep = final_mask & (~keep_mask)
        ax = axes[row_idx, col_idx]
        ax.imshow(common._robust_rescale(dapi_raw), cmap="gray")
        overlay = np.zeros((*dapi_raw.shape, 4), dtype=np.float32)
        overlay[removed_sweep, 0] = 1.0
        overlay[removed_sweep, 2] = 1.0
        overlay[removed_sweep, 3] = 0.80
        ax.imshow(overlay)
        _mask_contour(ax, final_mask, color="cyan", lw=0.7)
        _mask_contour(ax, analysis_mask_sweep, color="lime", lw=0.8)
        _scatter_beads(ax, beads_xy, color="magenta")
        removed_frac = float(np.sum(removed_sweep) / max(np.sum(final_mask), 1))
        ax.set_title(f"{pos} | N={n_sigma:.0f} sigma\nremoved {100.0 * removed_frac:.1f}%")
        ax.axis("off")

fig.savefig(FIXED_DAPI_GATE_SWEEP_PNG, dpi=180, bbox_inches="tight")
plt.show()

removed_value_limits = []
for pos in DAPI_GATE_SWEEP_POSITIONS:
    payload = analysis_payloads[str(pos)]
    dapi_raw = np.asarray(payload["dapi_raw"], dtype=np.float32)
    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    finite_all = np.isfinite(dapi_raw)
    for n_sigma in DAPI_GATE_SWEEP_N_SIGMA:
        threshold = float(bg_mu_map["dapi"] + float(n_sigma) * bg_sigma_map["dapi"])
        keep_mask = finite_all & (dapi_raw >= threshold)
        removed_sweep = final_mask & (~keep_mask)
        vals = dapi_raw[removed_sweep]
        vals = vals[np.isfinite(vals)]
        if vals.size:
            removed_value_limits.append(vals)
concat_removed = np.concatenate(removed_value_limits) if removed_value_limits else np.array([0.0, 1.0], dtype=np.float32)
removed_lo = float(np.nanquantile(concat_removed, 0.01))
removed_hi = float(np.nanquantile(concat_removed, 0.99))
if not np.isfinite(removed_lo):
    removed_lo = float(np.nanmin(concat_removed))
if not np.isfinite(removed_hi) or removed_hi <= removed_lo:
    removed_hi = removed_lo + 1.0

fig, axes = plt.subplots(
    len(DAPI_GATE_SWEEP_POSITIONS),
    len(DAPI_GATE_SWEEP_N_SIGMA),
    figsize=(3.2 * len(DAPI_GATE_SWEEP_N_SIGMA), 3.8 * len(DAPI_GATE_SWEEP_POSITIONS)),
    constrained_layout=True,
)
if len(DAPI_GATE_SWEEP_POSITIONS) == 1:
    axes = np.asarray([axes])
if len(DAPI_GATE_SWEEP_N_SIGMA) == 1:
    axes = axes[:, np.newaxis]

for row_idx, pos in enumerate(DAPI_GATE_SWEEP_POSITIONS):
    payload = analysis_payloads[str(pos)]
    dapi_raw = np.asarray(payload["dapi_raw"], dtype=np.float32)
    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    finite_all = np.isfinite(dapi_raw)
    beads_xy = payload["beads_xy_display"]
    for col_idx, n_sigma in enumerate(DAPI_GATE_SWEEP_N_SIGMA):
        threshold = float(bg_mu_map["dapi"] + float(n_sigma) * bg_sigma_map["dapi"])
        keep_mask = finite_all & (dapi_raw >= threshold)
        analysis_mask_sweep = final_mask & keep_mask
        removed_sweep = final_mask & (~keep_mask)
        removed_values = np.where(removed_sweep, dapi_raw, np.nan)
        ax = axes[row_idx, col_idx]
        ax.imshow(removed_values, cmap="inferno", vmin=removed_lo, vmax=removed_hi)
        _mask_contour(ax, final_mask, color="cyan", lw=0.7)
        _mask_contour(ax, analysis_mask_sweep, color="lime", lw=0.8)
        _scatter_beads(ax, beads_xy, color="magenta")
        removed_frac = float(np.sum(removed_sweep) / max(np.sum(final_mask), 1))
        ax.set_title(f"{pos} | N={n_sigma:.0f} sigma\nremoved DAPI values")
        ax.text(
            0.98,
            0.04,
            f"{100.0 * removed_frac:.1f}% removed",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8,
            color="white",
            bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55, "boxstyle": "round,pad=0.2"},
        )
        ax.axis("off")

fig.savefig(FIXED_DAPI_GATE_SWEEP_VALUES_PNG, dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
removed_limits = {}
for channel_key, arr_name in [
    ("dapi", "dapi_raw"),
    ("tagyfp", "tagyfp_raw"),
    ("sox2", "sox2_raw"),
    ("t", "t_raw"),
]:
    vals_all = []
    for pos in REPRESENTATIVE_POSITIONS:
        payload = analysis_payloads[str(pos)]
        removed = payload["removed_by_dapi_gate"]
        vals = np.asarray(payload[arr_name][removed], dtype=np.float32)
        vals = vals[np.isfinite(vals)]
        if vals.size:
            vals_all.append(vals)
    concat = np.concatenate(vals_all) if vals_all else np.array([0.0], dtype=np.float32)
    lo = float(np.nanquantile(concat, 0.01))
    hi = float(np.nanquantile(concat, 0.99))
    if not np.isfinite(lo):
        lo = float(np.nanmin(concat))
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    removed_limits[channel_key] = (lo, hi)

fig, axes = plt.subplots(len(REPRESENTATIVE_POSITIONS), 8, figsize=(28.0, 3.9 * len(REPRESENTATIVE_POSITIONS)), constrained_layout=True)
if len(REPRESENTATIVE_POSITIONS) == 1:
    axes = np.asarray([axes])
for row_idx, pos in enumerate(REPRESENTATIVE_POSITIONS):
    payload = analysis_payloads[str(pos)]
    dapi_raw = payload["dapi_raw"]
    final_mask = payload["final_mask"]
    removed = payload["removed_by_dapi_gate"]
    analysis_mask = payload["analysis_mask"]
    beads_xy = payload["beads_xy_display"]

    ax = axes[row_idx, 0]
    ax.imshow(common._robust_rescale(dapi_raw), cmap="gray")
    _mask_contour(ax, final_mask, color="cyan")
    _scatter_beads(ax, beads_xy, color="magenta")
    ax.set_title(f"{pos} | Raw DAPI + final mask")
    ax.axis("off")

    ax = axes[row_idx, 1]
    ax.imshow(common._robust_rescale(dapi_raw), cmap="gray")
    overlay = np.zeros((*dapi_raw.shape, 4), dtype=np.float32)
    overlay[removed, 0] = 1.0
    overlay[removed, 2] = 1.0
    overlay[removed, 3] = 0.85
    ax.imshow(overlay)
    _mask_contour(ax, final_mask, color="cyan")
    ax.set_title(f"{pos} | Pixels removed by DAPI gate")
    ax.axis("off")

    ax = axes[row_idx, 2]
    ax.imshow(removed.astype(np.float32), cmap="magma")
    ax.set_title(f"{pos} | Removed pixels only")
    ax.axis("off")

    for col_idx, (title, arr_name, cmap_key) in enumerate([
        ("Removed DAPI values", "dapi_raw", "dapi"),
        ("Removed FOXF1-YFP values", "tagyfp_raw", "tagyfp"),
        ("Removed SOX2 values", "sox2_raw", "sox2"),
        ("Removed T values", "t_raw", "t"),
    ], start=3):
        ax = axes[row_idx, col_idx]
        arr = np.asarray(payload[arr_name], dtype=np.float32)
        lo, hi = removed_limits[cmap_key]
        show = np.where(removed, arr, np.nan)
        ax.imshow(show, cmap="inferno", vmin=lo, vmax=hi)
        ax.set_title(f"{pos} | {title}")
        ax.axis("off")

    ax = axes[row_idx, 7]
    ax.imshow(common._robust_rescale(dapi_raw), cmap="gray")
    _mask_contour(ax, analysis_mask, color="lime")
    _scatter_beads(ax, beads_xy, color="magenta")
    ax.set_title(f"{pos} | Final analysis mask")
    ax.axis("off")

fig.savefig(FIXED_DAPI_GATE_DEBUG_PNG, dpi=180, bbox_inches="tight")
plt.show()

## Applied Exclusion Summary

This downstream QC shows how the already-approved manual exclusion ROI intersects the DAPI-gated channel-specific analysis masks after background fitting and gating.

Use this section to confirm the applied exclusion footprint, not to decide whether an exclusion should exist.

In [ ]:
channel_labels = {
    "tagyfp": "FOXF1-YFP (fixed)",
    "sox2": "SOX2",
    "t": "T",
}
raw_keys = {
    "tagyfp": "tagyfp_raw",
    "sox2": "sox2_raw",
    "t": "t_raw",
}
bgz_keys = {
    "tagyfp": "tagyfp_bgz",
    "sox2": "sox2_bgz",
    "t": "t_bgz",
}
analysis_mask_keys = {
    "tagyfp": "tagyfp_analysis_mask",
    "sox2": "sox2_analysis_mask",
    "t": "t_analysis_mask",
}
manual_mask_keys = {
    "tagyfp": "tagyfp_manual_exclusion_mask",
    "sox2": "sox2_manual_exclusion_mask",
    "t": "t_manual_exclusion_mask",
}
manual_reason_keys = {
    "tagyfp": "tagyfp_manual_exclusion_reasons",
    "sox2": "sox2_manual_exclusion_reasons",
    "t": "t_manual_exclusion_reasons",
}

manual_rows = []
for pos in sorted(analysis_payloads):
    payload = analysis_payloads[str(pos)]
    analysis_mask = np.asarray(payload["analysis_mask"], dtype=bool)
    for channel_key in ["tagyfp", "sox2", "t"]:
        manual_mask = np.asarray(payload[manual_mask_keys[channel_key]], dtype=bool)
        if not np.any(manual_mask):
            continue
        channel_analysis_mask = np.asarray(payload[analysis_mask_keys[channel_key]], dtype=bool)
        manual_rows.append(
            {
                "canonical_position": str(pos),
                "channel_key": channel_key,
                "channel_label": channel_labels[channel_key],
                "analysis_mask_area_px": int(np.sum(analysis_mask)),
                "manual_exclusion_px": int(np.sum(manual_mask)),
                "channel_analysis_mask_area_px": int(np.sum(channel_analysis_mask)),
                "manual_exclusion_fraction_within_analysis_mask": float(np.sum(manual_mask) / max(np.sum(analysis_mask), 1)),
                "manual_exclusion_reasons": ";".join(payload.get(manual_reason_keys[channel_key], [])),
            }
        )

manual_exclusion_summary_df = pd.DataFrame(manual_rows).sort_values(["channel_key", "canonical_position"]).reset_index(drop=True)
manual_exclusion_summary_df.to_csv(FIXED_MANUAL_EXCLUSION_SUMMARY_TSV, sep="	", index=False)
display(manual_exclusion_summary_df)

manual_entries = manual_exclusion_summary_df[["canonical_position", "channel_key"]].drop_duplicates().to_dict("records") if len(manual_exclusion_summary_df) else []
if manual_entries:
    channel_limits = {}
    for channel_key in ["tagyfp", "sox2", "t"]:
        vals_all = []
        for entry in manual_entries:
            if entry["channel_key"] != channel_key:
                continue
            payload = analysis_payloads[str(entry["canonical_position"])]
            vals = np.asarray(payload[bgz_keys[channel_key]][payload["analysis_mask"]], dtype=np.float32)
            vals = vals[np.isfinite(vals)]
            if vals.size:
                vals_all.append(vals)
        concat = np.concatenate(vals_all) if vals_all else np.array([0.0], dtype=np.float32)
        lim = max(abs(float(np.nanquantile(concat, 0.01))), abs(float(np.nanquantile(concat, 0.99))))
        channel_limits[channel_key] = max(lim, 1e-6)

    fig, axes = plt.subplots(len(manual_entries), 7, figsize=(25.5, 3.8 * len(manual_entries)), constrained_layout=True)
    if len(manual_entries) == 1:
        axes = np.asarray([axes])
    for row_idx, entry in enumerate(manual_entries):
        pos = str(entry["canonical_position"])
        channel_key = str(entry["channel_key"])
        channel_label = channel_labels[channel_key]
        payload = analysis_payloads[pos]
        dapi_raw = np.asarray(payload["dapi_raw"], dtype=np.float32)
        raw_arr = np.asarray(payload[raw_keys[channel_key]], dtype=np.float32)
        bgz_arr = np.asarray(payload[bgz_keys[channel_key]], dtype=np.float32)
        analysis_mask = np.asarray(payload["analysis_mask"], dtype=bool)
        manual_mask = np.asarray(payload[manual_mask_keys[channel_key]], dtype=bool)
        channel_analysis_mask = np.asarray(payload[analysis_mask_keys[channel_key]], dtype=bool)
        beads_xy = np.asarray(payload["beads_xy_display"], dtype=np.float32)
        lim = float(channel_limits[channel_key])

        ax = axes[row_idx, 0]
        ax.imshow(common._robust_rescale(dapi_raw), cmap="gray")
        _mask_contour(ax, analysis_mask, color="lime")
        _scatter_beads(ax, beads_xy, color="magenta")
        ax.set_title(f"{pos} | Raw DAPI + base analysis mask")
        ax.axis("off")

        ax = axes[row_idx, 1]
        ax.imshow(common._robust_rescale(raw_arr), cmap="gray")
        _mask_contour(ax, analysis_mask, color="white", lw=0.7)
        ax.set_title(f"{pos} | {channel_label} raw")
        ax.axis("off")

        ax = axes[row_idx, 2]
        ax.imshow(common._robust_rescale(raw_arr), cmap="gray")
        overlay = np.zeros((*raw_arr.shape, 4), dtype=np.float32)
        overlay[manual_mask, 0] = 1.0
        overlay[manual_mask, 2] = 1.0
        overlay[manual_mask, 3] = 0.85
        ax.imshow(overlay)
        _mask_contour(ax, analysis_mask, color="white", lw=0.7)
        ax.set_title(f"{pos} | {channel_label} raw + manual exclusion")
        ax.axis("off")

        ax = axes[row_idx, 3]
        show = np.where(analysis_mask, bgz_arr, np.nan)
        ax.imshow(show, cmap="RdBu_r", vmin=-lim, vmax=lim)
        _mask_contour(ax, analysis_mask, color="white", lw=0.7)
        ax.set_title(f"{pos} | {channel_label} bgz")
        ax.axis("off")

        ax = axes[row_idx, 4]
        ax.imshow(show, cmap="RdBu_r", vmin=-lim, vmax=lim)
        overlay = np.zeros((*bgz_arr.shape, 4), dtype=np.float32)
        overlay[manual_mask, 0] = 1.0
        overlay[manual_mask, 2] = 1.0
        overlay[manual_mask, 3] = 0.85
        ax.imshow(overlay)
        _mask_contour(ax, analysis_mask, color="white", lw=0.7)
        ax.set_title(f"{pos} | {channel_label} bgz + manual exclusion")
        ax.axis("off")

        ax = axes[row_idx, 5]
        removed_vals = np.where(manual_mask, bgz_arr, np.nan)
        ax.imshow(removed_vals, cmap="inferno")
        ax.set_title(f"{pos} | Removed {channel_label} bgz pixels")
        ax.axis("off")

        ax = axes[row_idx, 6]
        ax.imshow(np.where(channel_analysis_mask, bgz_arr, np.nan), cmap="RdBu_r", vmin=-lim, vmax=lim)
        _mask_contour(ax, channel_analysis_mask, color="lime", lw=0.8)
        _scatter_beads(ax, beads_xy, color="magenta")
        ax.set_title(f"{pos} | Final {channel_label} analysis mask")
        ax.axis("off")

    fig.savefig(FIXED_MANUAL_EXCLUSION_QC_PNG, dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("No manual channel exclusions configured.")

## Fixed Ratio Image Diagnostics

In [ ]:
bgz_limits = {}
for channel_key, arr_name in [("tagyfp", "tagyfp_bgz"), ("sox2", "sox2_bgz"), ("t", "t_bgz")]:
    vals_all = []
    for pos in REPRESENTATIVE_POSITIONS:
        payload = analysis_payloads[str(pos)]
        use_mask = payload[f"{channel_key}_analysis_mask"]
        vals = np.asarray(payload[arr_name][use_mask], dtype=np.float32)
        vals = vals[np.isfinite(vals)]
        if vals.size:
            vals_all.append(vals)
    concat = np.concatenate(vals_all) if vals_all else np.array([0.0], dtype=np.float32)
    lim = max(abs(float(np.nanquantile(concat, 0.01))), abs(float(np.nanquantile(concat, 0.99))))
    bgz_limits[channel_key] = max(lim, 1e-6)

ratio_limits = {}
for meas_name, arr_name, channel_key in [
    ("fixed_tagyfp_bgz_over_dapi_gate", "tagyfp_ratio", "tagyfp"),
    ("fixed_sox2_bgz_over_dapi_gate", "sox2_ratio", "sox2"),
    ("fixed_t_bgz_over_dapi_gate", "t_ratio", "t"),
]:
    vals_all = []
    for pos in REPRESENTATIVE_POSITIONS:
        payload = analysis_payloads[str(pos)]
        use_mask = payload[f"{channel_key}_analysis_mask"]
        vals = np.asarray(payload[arr_name][use_mask], dtype=np.float32)
        vals = vals[np.isfinite(vals)]
        if vals.size:
            vals_all.append(vals)
    concat = np.concatenate(vals_all) if vals_all else np.array([0.0], dtype=np.float32)
    lim = max(abs(float(np.nanquantile(concat, 0.01))), abs(float(np.nanquantile(concat, 0.99))))
    ratio_limits[meas_name] = max(lim, 1e-6)

fig, axes = plt.subplots(len(REPRESENTATIVE_POSITIONS), 8, figsize=(26.0, 3.8 * len(REPRESENTATIVE_POSITIONS)), constrained_layout=True)
if len(REPRESENTATIVE_POSITIONS) == 1:
    axes = np.asarray([axes])
for row_idx, pos in enumerate(REPRESENTATIVE_POSITIONS):
    payload = analysis_payloads[str(pos)]
    dapi_raw = payload["dapi_raw"]
    analysis_mask = payload["analysis_mask"]
    tagyfp_analysis_mask = payload["tagyfp_analysis_mask"]
    sox2_analysis_mask = payload["sox2_analysis_mask"]
    t_analysis_mask = payload["t_analysis_mask"]
    beads_xy = payload["beads_xy_display"]
    dist_um = payload["distance_um"]

    ax = axes[row_idx, 0]
    ax.imshow(common._robust_rescale(dapi_raw), cmap="gray")
    _mask_contour(ax, analysis_mask, color="lime")
    _scatter_beads(ax, beads_xy, color="magenta")
    ax.set_title(f"{pos} | Raw DAPI + analysis mask")
    ax.axis("off")

    ax = axes[row_idx, 1]
    tagyfp_bgz_show = np.where(tagyfp_analysis_mask, payload["tagyfp_bgz"], np.nan)
    ax.imshow(tagyfp_bgz_show, cmap="RdBu_r", vmin=-bgz_limits["tagyfp"], vmax=bgz_limits["tagyfp"])
    _mask_contour(ax, tagyfp_analysis_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | FOXF1-YFP bgz")
    ax.axis("off")

    ax = axes[row_idx, 2]
    sox2_bgz_show = np.where(sox2_analysis_mask, payload["sox2_bgz"], np.nan)
    ax.imshow(sox2_bgz_show, cmap="RdBu_r", vmin=-bgz_limits["sox2"], vmax=bgz_limits["sox2"])
    _mask_contour(ax, sox2_analysis_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | SOX2 bgz")
    ax.axis("off")

    ax = axes[row_idx, 3]
    t_bgz_show = np.where(t_analysis_mask, payload["t_bgz"], np.nan)
    ax.imshow(t_bgz_show, cmap="RdBu_r", vmin=-bgz_limits["t"], vmax=bgz_limits["t"])
    _mask_contour(ax, t_analysis_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | T bgz")
    ax.axis("off")

    ax = axes[row_idx, 4]
    tagyfp_ratio_show = np.where(tagyfp_analysis_mask, payload["tagyfp_ratio"], np.nan)
    ax.imshow(tagyfp_ratio_show, cmap="RdBu_r", vmin=-ratio_limits["fixed_tagyfp_bgz_over_dapi_gate"], vmax=ratio_limits["fixed_tagyfp_bgz_over_dapi_gate"])
    _mask_contour(ax, tagyfp_analysis_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | FOXF1-YFP bgz / DAPI")
    ax.axis("off")

    ax = axes[row_idx, 5]
    sox2_ratio_show = np.where(sox2_analysis_mask, payload["sox2_ratio"], np.nan)
    ax.imshow(sox2_ratio_show, cmap="RdBu_r", vmin=-ratio_limits["fixed_sox2_bgz_over_dapi_gate"], vmax=ratio_limits["fixed_sox2_bgz_over_dapi_gate"])
    _mask_contour(ax, sox2_analysis_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | SOX2 bgz / DAPI")
    ax.axis("off")

    ax = axes[row_idx, 6]
    t_ratio_show = np.where(t_analysis_mask, payload["t_ratio"], np.nan)
    ax.imshow(t_ratio_show, cmap="RdBu_r", vmin=-ratio_limits["fixed_t_bgz_over_dapi_gate"], vmax=ratio_limits["fixed_t_bgz_over_dapi_gate"])
    _mask_contour(ax, t_analysis_mask, color="white", lw=0.7)
    ax.set_title(f"{pos} | T bgz / DAPI")
    ax.axis("off")

    ax = axes[row_idx, 7]
    ax.imshow(dist_um, cmap="viridis")
    _mask_contour(ax, analysis_mask, color="white", lw=0.7)
    _scatter_beads(ax, beads_xy, color="magenta")
    ax.set_title(f"{pos} | Distance from nearest bead")
    ax.axis("off")

fig.savefig(FIXED_RATIO_REPRESENTATIVE_PNG, dpi=180, bbox_inches="tight")
plt.show()

## Fixed Distance Quantification Outputs

In [ ]:
fixed_ratio_stats_df = pd.DataFrame(ratio_rows).sort_values(["measurement_name", "canonical_position", "bin_idx"]).reset_index(drop=True)
fixed_ratio_stats_df.to_csv(FIXED_RATIO_PIXEL_BIN_STATS_TSV, sep="\t", index=False)

fixed_ratio_trace_df = aggregate_trace_across_images(fixed_ratio_stats_df)
fixed_ratio_trace_df.to_csv(FIXED_RATIO_TRACE_ACROSS_IMAGES_TSV, sep="\t", index=False)

fixed_ratio_merged_df = weighted_trace_pixels(fixed_ratio_stats_df)
fixed_ratio_merged_df.to_csv(FIXED_RATIO_TRACE_ALL_PIXELS_TSV, sep="\t", index=False)

if len(fixed_ratio_merged_df) == 0:
    equal_support_target_px = 1
else:
    shared_support = fixed_ratio_merged_df[fixed_ratio_merged_df["measurement_name"] == MEASUREMENT_ORDER[0]]["total_count_px"].astype(float)
    equal_support_target_px = int(np.nanmedian(shared_support)) if len(shared_support) else int(np.nanmedian(fixed_ratio_merged_df["total_count_px"].astype(float)))

equal_support_tables = []
equal_support_agg_tables = []
for meas_name in MEASUREMENT_ORDER:
    payloads = pixel_payloads_by_measurement[meas_name]
    if not payloads:
        continue
    dist_all = np.concatenate([np.asarray(v["distance_um"], dtype=np.float32) for v in payloads.values() if len(v["distance_um"])])
    val_all = np.concatenate([np.asarray(v["value"], dtype=np.float32) for v in payloads.values() if len(v["value"])])
    eq_df = equal_support_trace(
        dist_all,
        val_all,
        pixels_per_bin=equal_support_target_px,
        measurement_name=meas_name,
    )
    equal_support_tables.append(eq_df)
    equal_support_agg_tables.append(
        aggregate_equal_support_across_images(
            payloads=payloads,
            equal_support_df=eq_df,
            measurement_name=meas_name,
        )
    )

fixed_ratio_equal_support_df = pd.concat(equal_support_tables, ignore_index=True) if equal_support_tables else pd.DataFrame()
fixed_ratio_equal_support_df.to_csv(FIXED_RATIO_TRACE_EQUAL_SUPPORT_TSV, sep="\t", index=False)

fixed_ratio_equal_support_across_images_df = pd.concat(equal_support_agg_tables, ignore_index=True) if equal_support_agg_tables else pd.DataFrame()
fixed_ratio_equal_support_across_images_df.to_csv(FIXED_RATIO_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="\t", index=False)

channel_manual_positions = {}
for channel_key in ["tagyfp", "sox2", "t"]:
    pos_list = sorted(
        [
            str(pos)
            for pos, payload in analysis_payloads.items()
            if np.any(np.asarray(payload[f"{channel_key}_manual_exclusion_mask"], dtype=bool))
        ]
    )
    channel_manual_positions[channel_key] = pos_list

summary_txt = (
    f"positions_ok\t{len(analysis_payloads)}\n"
    f"dapi_gate_threshold\t{dapi_gate_threshold:.6f}\n"
    f"dapi_gate_n_sigma\t{DAPI_GATE_N_SIGMA:.1f}\n"
    f"distance_bin_um\t{DIST_BIN_UM:.1f}\n"
    f"equal_support_target_pixels\t{int(equal_support_target_px)}\n"
    f"tagyfp_manual_exclusion_positions\t{';'.join(channel_manual_positions['tagyfp'])}\n"
    f"sox2_manual_exclusion_positions\t{';'.join(channel_manual_positions['sox2'])}\n"
    f"t_manual_exclusion_positions\t{';'.join(channel_manual_positions['t'])}\n"
    f"fixed_tagyfp_position_exclusions\t{';'.join(sorted(measurement_position_exclusions.get('fixed_tagyfp_bgz_over_dapi_gate', set())))}\n"
    f"measurement_strategy\tglobal_raw_channel_bg_zscore_over_raw_dapi_inside_final04_mask_and_pooled_raw_dapi_gate_with_channel_specific_manual_exclusion_applied_before_bgfit_and_before_ratio_measurement_and_with_optional_measurement_level_scene_exclusion\n"
)
FIXED_MEASUREMENT_SUMMARY_TXT.write_text(summary_txt)

FOXF1_FIXED_MEAS_NAME = "fixed_tagyfp_bgz_over_dapi_gate"

foxf1_stats_df = fixed_ratio_stats_df[fixed_ratio_stats_df["measurement_name"] == FOXF1_FIXED_MEAS_NAME].copy()
foxf1_stats_df = foxf1_stats_df.sort_values(["canonical_position", "bin_idx"]).reset_index(drop=True)
foxf1_stats_df.to_csv(FIXED_FOXF1_PIXEL_BIN_STATS_TSV, sep="\t", index=False)

foxf1_trace_df = fixed_ratio_trace_df[fixed_ratio_trace_df["measurement_name"] == FOXF1_FIXED_MEAS_NAME].copy()
foxf1_trace_df = foxf1_trace_df.sort_values("bin_mid_um").reset_index(drop=True)
foxf1_trace_df.to_csv(FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV, sep="\t", index=False)

foxf1_merged_df = fixed_ratio_merged_df[fixed_ratio_merged_df["measurement_name"] == FOXF1_FIXED_MEAS_NAME].copy()
foxf1_merged_df = foxf1_merged_df.sort_values("bin_mid_um").reset_index(drop=True)
foxf1_merged_df.to_csv(FIXED_FOXF1_TRACE_ALL_PIXELS_TSV, sep="\t", index=False)

foxf1_equal_support_df = fixed_ratio_equal_support_df[fixed_ratio_equal_support_df["measurement_name"] == FOXF1_FIXED_MEAS_NAME].copy()
foxf1_equal_support_df = foxf1_equal_support_df.sort_values("bin_mid_um").reset_index(drop=True)
foxf1_equal_support_df.to_csv(FIXED_FOXF1_TRACE_EQUAL_SUPPORT_TSV, sep="\t", index=False)

foxf1_equal_support_across_images_df = fixed_ratio_equal_support_across_images_df[
    fixed_ratio_equal_support_across_images_df["measurement_name"] == FOXF1_FIXED_MEAS_NAME
].copy()
foxf1_equal_support_across_images_df = foxf1_equal_support_across_images_df.sort_values("bin_mid_um").reset_index(drop=True)
foxf1_equal_support_across_images_df.to_csv(FIXED_FOXF1_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="\t", index=False)

pooled_ratio_arrays = {}
foxf1_vals_all = []
sox2_joint_vals_all = []
sox2_vals_all = []
sox2_t_pair_vals_all = []
t_vals_all = []
t_pair_vals_all = []
for pos in sorted(analysis_payloads):
    payload = analysis_payloads[str(pos)]
    tagyfp_analysis_mask = np.asarray(payload["tagyfp_analysis_mask"], dtype=bool)
    sox2_analysis_mask = np.asarray(payload["sox2_analysis_mask"], dtype=bool)
    t_analysis_mask = np.asarray(payload["t_analysis_mask"], dtype=bool)
    foxf1_scene_excluded = str(pos) in measurement_position_exclusions.get('fixed_tagyfp_bgz_over_dapi_gate', set())

    if not foxf1_scene_excluded:
        foxf1_vals = np.asarray(payload["tagyfp_ratio"][tagyfp_analysis_mask], dtype=np.float32)
        foxf1_vals = foxf1_vals[np.isfinite(foxf1_vals)]
        if foxf1_vals.size:
            foxf1_vals_all.append(foxf1_vals)

        foxf1_sox2_pair_mask = tagyfp_analysis_mask & sox2_analysis_mask
        sox2_joint_vals = np.asarray(payload["sox2_ratio"][foxf1_sox2_pair_mask], dtype=np.float32)
        sox2_joint_vals = sox2_joint_vals[np.isfinite(sox2_joint_vals)]
        if sox2_joint_vals.size:
            sox2_joint_vals_all.append(sox2_joint_vals)

    sox2_vals = np.asarray(payload["sox2_ratio"][sox2_analysis_mask], dtype=np.float32)
    sox2_vals = sox2_vals[np.isfinite(sox2_vals)]
    if sox2_vals.size:
        sox2_vals_all.append(sox2_vals)

    sox2_t_pair_mask = sox2_analysis_mask & t_analysis_mask
    sox2_t_pair_vals = np.asarray(payload["sox2_ratio"][sox2_t_pair_mask], dtype=np.float32)
    sox2_t_pair_vals = sox2_t_pair_vals[np.isfinite(sox2_t_pair_vals)]
    if sox2_t_pair_vals.size:
        sox2_t_pair_vals_all.append(sox2_t_pair_vals)

    t_vals = np.asarray(payload["t_ratio"][t_analysis_mask], dtype=np.float32)
    t_vals = t_vals[np.isfinite(t_vals)]
    if t_vals.size:
        t_vals_all.append(t_vals)

    t_pair_vals = np.asarray(payload["t_ratio"][sox2_t_pair_mask], dtype=np.float32)
    t_pair_vals = t_pair_vals[np.isfinite(t_pair_vals)]
    if t_pair_vals.size:
        t_pair_vals_all.append(t_pair_vals)

pooled_ratio_arrays["foxf1_ratio"] = np.concatenate(foxf1_vals_all) if foxf1_vals_all else np.array([], dtype=np.float32)
pooled_ratio_arrays["sox2_ratio_for_foxf1_pair"] = np.concatenate(sox2_joint_vals_all) if sox2_joint_vals_all else np.array([], dtype=np.float32)
pooled_ratio_arrays["sox2_ratio"] = np.concatenate(sox2_vals_all) if sox2_vals_all else np.array([], dtype=np.float32)
pooled_ratio_arrays["sox2_ratio_for_t_pair"] = np.concatenate(sox2_t_pair_vals_all) if sox2_t_pair_vals_all else np.array([], dtype=np.float32)
pooled_ratio_arrays["t_ratio"] = np.concatenate(t_vals_all) if t_vals_all else np.array([], dtype=np.float32)
pooled_ratio_arrays["t_ratio_for_sox2_pair"] = np.concatenate(t_pair_vals_all) if t_pair_vals_all else np.array([], dtype=np.float32)
np.savez_compressed(FIXED_RATIO_ANALYSIS_PIXELS_NPZ, **pooled_ratio_arrays)

print("Saved fixed quantification intermediates:")
print("  ", FIXED_RATIO_PIXEL_BIN_STATS_TSV.name)
print("  ", FIXED_RATIO_TRACE_ACROSS_IMAGES_TSV.name)
print("  ", FIXED_RATIO_TRACE_ALL_PIXELS_TSV.name)
print("  ", FIXED_RATIO_TRACE_EQUAL_SUPPORT_TSV.name)
print("  ", FIXED_RATIO_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV.name)
print("  ", FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV.name)
print("  ", FIXED_FOXF1_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV.name)
print("  ", FIXED_RATIO_ANALYSIS_PIXELS_NPZ.name)
print()
print("Fixed ratio stats head:")
display(fixed_ratio_stats_df.head())
print("FOXF1 fixed trace head:")
display(foxf1_trace_df.head())


## Downstream Fixed Review And Plotting

The focused FOXF1 dip diagnosis now lives in [08_fixed_foxf1_dip_review.ipynb](notebooks/08_fixed_foxf1_dip_review.ipynb). The broader fixed-distance plotting notebook now lives in [09_fixed_plotting.ipynb](notebooks/09_fixed_plotting.ipynb). This notebook writes the intermediates that both downstream notebooks consume.